Be prepared for failure and retries:
- llms don't always do as their told (for example json might be perfect)
- llms services might not be available perfectly (like OpenAI)
> Classic track note: This notebook demonstrates legacy/classic LangChain-era patterns for evaluation and comparison.
> Prefer the modern equivalents in `lessons/2026/langchain/` for current APIs and recommended techniques.


In [ ]:
%pip install -q langchain langchain-openai

In [ ]:
%pip install -q python-dotenv
from dotenv import load_dotenv
load_dotenv()

We can use a parser to check the output of the llm and we can also use it to provide hints to the llm to improve it's output.

In [ ]:
# https://python.langchain.com/docs/modules/model_io/output_parsers/retry
# https://python.langchain.com/docs/modules/model_io/output_parsers/structured

from langchain.prompts import (
    PromptTemplate,
)
from langchain_openai import OpenAI, ChatOpenAI
from langchain.output_parsers import (
    PydanticOutputParser,
    OutputFixingParser,
    RetryOutputParser,
)
from pydantic import BaseModel, Field, validator
from typing import List


For example we specify the schema of the output we expected

In [ ]:

template = """Based on the user question, provide an Action and Action Input for what step should be taken.
{format_instructions}
Question: {query}
Response:"""

class Action(BaseModel):
    action: str = Field(description="action to take")
    action_input: str = Field(description="input to the action")


parser = PydanticOutputParser(pydantic_object=Action)

In [ ]:
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

Here's an example of something that would be bad as we parse it.
It misses one of the two fields in the response.

In [ ]:
prompt_value = prompt.format_prompt(query="who is leo di caprios gf?")

bad_response = "{\"action\": \"search\"}"
try:
    parser.parse(bad_response)
except Exception as exc:
    print(f"Expected parser failure for malformed response: {exc}")


Now we give the llm hints through the parse instructions we described.
See how it uses that under the hood.

In [ ]:
import sys
sys.path.append('../developer')

from _lessonshelper.pretty_print_callback_handler import PrettyPrintCallbackHandler
callback = PrettyPrintCallbackHandler()


fix_parser = OutputFixingParser.from_llm(parser=parser, llm=ChatOpenAI(callbacks=[callback]))
fix_parser.parse(bad_response)


Now that we have the instructions, we can have it retry that automatically if it doesn't comply.

In [ ]:
from langchain.output_parsers import RetryWithErrorOutputParser

retry_parser = RetryWithErrorOutputParser.from_llm(
    parser=parser, llm=OpenAI(temperature=0, callbacks=[callback])
)

retry_parser.parse_with_prompt(bad_response, prompt_value)

